# KIRBy: Get a Powerful Hybrid Representation for Your Dataset

KIRBy searches over representation combinations, feature selection strategies, and budgets to find the best hybrid representation for your task. This notebook shows three workflows:

1. **Full search** — run `find_best_hybrid` on your own data
2. **Published lookup** — use a pre-computed best hybrid for a known dataset
3. **Apply a blueprint** — take a published recipe and apply it to new data

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'tests'))
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

import numpy as np
from kirby.find_best_hybrid import find_best_hybrid, SearchResult
from kirby.hybrid import create_hybrid, apply_feature_selection

## 1. Full Search: Find the Best Hybrid for Your Dataset

Load your data and representations, then run `find_best_hybrid`.

In [ ]:
# Load a dataset using KIRBy's built-in loaders
from experiments._shared import load_data, SCREENING_BUDGETS

dataset_name = 'logd'  # or: herg, caco2, flab_thermo_tresanco, etc.
data = load_data(dataset_name, rep_preset='all')

reps = data['reps']      # {name: np.ndarray} — each (n_samples, n_features)
labels = data['labels']  # (n_samples,)
groups = data['groups']  # (n_samples,) — scaffold/cluster groups for CV

print(f"Dataset: {dataset_name}")
print(f"Samples: {len(labels)}")
print(f"Representations: {len(reps)}")
for name, X in sorted(reps.items()):
    print(f"  {name}: {X.shape[1]}D")

In [ ]:
# Run the hybrid search
# speed='fast' is good for screening; 'full' for publication-quality
result = find_best_hybrid(
    base_reps=reps,
    labels=labels,
    groups=groups,
    budgets=SCREENING_BUDGETS,  # [50, 500, 1000, 2000, 3000, 4000]
    speed='fast',
    metric='r2',       # or: rmse, spearman, auc, f1_weighted, ci
    task_type='auto',  # auto-detects regression vs classification
    cache=True,        # cache results to ~/.cache/kirby/hybrid_search/
    seed=42,
)

In [ ]:
# Best overall configuration
best = result.best_overall
print(f"Best score: {best['score_mean']:.4f} +/- {best['score_std']:.3f}")
print(f"Model: {best['model']}")
print(f"Budget: {best['budget']}")
print(f"Allocation: {best['allocation_method']}")
print(f"Importance: {best['importance_method']}")
print(f"Features: {best['n_features']}")

In [ ]:
# Best by budget — pick the smallest budget that's nearly as good
print(f"\n{'Budget':>7s}  {'Score':>10s}  {'Model':<10s}  {'Allocation':<20s}")
print("-" * 55)

best_score = best['score_mean']
is_lower_better = result.lower_is_better
for bgt in sorted(result.best_by_budget.keys()):
    r = result.best_by_budget[bgt]
    raw = r['score_mean']
    if best_score > 0 and raw > 0:
        pct = (best_score / raw * 100) if is_lower_better else (raw / best_score * 100)
        pct_str = f"({pct:.1f}%)"
    else:
        delta = abs(raw - best_score)
        pct_str = "(= best)" if delta < 1e-6 else f"(delta={delta:.4f})"
    marker = ' <-- best' if abs(raw - best_score) < 1e-6 else ''
    print(f"{bgt:>7d}  {raw:>10.4f}  {r['model']:<10s}  "
          f"{r['allocation_method']:<20s} {pct_str}{marker}")

In [ ]:
# Individual rep baselines (what you'd get without KIRBy)
print(f"\n{'Rep':<20s}  {'Dim':>5s}  {'RF':>8s}  {'LightGBM':>8s}  {'MLP':>8s}")
print("-" * 60)
for rep_name in sorted(result.baselines.keys()):
    bl = result.baselines[rep_name]
    dim = bl.get('dimension', '?')
    rf = bl.get('rf', float('nan'))
    lgb = bl.get('lightgbm', float('nan'))
    mlp = bl.get('mlp', float('nan'))
    print(f"{rep_name:<20s}  {dim:>5}  {rf:>8.4f}  {lgb:>8.4f}  {mlp:>8.4f}")

## 2. Published Lookup Table

If we've already run the search for your dataset, you can look up the best hybrid directly. The lookup table is built from cached search results.

In [ ]:
import pickle, json
from pathlib import Path

CACHE_DIR = Path.home() / '.cache' / 'kirby' / 'hybrid_search'

def load_all_cached_results(cache_dir=CACHE_DIR):
    """Load all cached SearchResult objects."""
    results = {}
    if not cache_dir.exists():
        print(f"No cache directory at {cache_dir}")
        return results
    for pkl_file in sorted(cache_dir.glob('*.pkl')):
        try:
            sr = pickle.loads(pkl_file.read_bytes())
            if hasattr(sr, 'best_overall') and sr.best_overall:
                key = pkl_file.stem  # cache hash
                results[key] = sr
        except Exception:
            continue
    return results


def build_lookup_table(cached_results):
    """Build a publishable lookup table from cached results.
    
    Returns list of dicts with: cache_key, metric, best_score, budget,
    model, allocation, importance, n_features, n_reps, blueprint.
    """
    rows = []
    for key, sr in cached_results.items():
        best = sr.best_overall
        bp = best['blueprint']
        row = {
            'cache_key': key,
            'metric': sr.metric_name,
            'best_score': round(best['score_mean'], 4),
            'score_std': round(best['score_std'], 4),
            'budget': best['budget'],
            'model': best['model'],
            'allocation': best['allocation_method'],
            'importance': best['importance_method'],
            'evaluator': best['internal_evaluator'],
            'n_features': best['n_features'],
            'n_reps': len(bp.reps),
            'reps': bp.reps,
            'blueprint': bp.to_dict(),
        }
        # Add best-by-budget breakdown
        by_budget = {}
        for bgt, r in getattr(sr, 'best_by_budget', {}).items():
            by_budget[bgt] = {
                'score': round(r['score_mean'], 4),
                'model': r['model'],
                'allocation': r['allocation_method'],
            }
        row['by_budget'] = by_budget
        rows.append(row)
    return rows


cached = load_all_cached_results()
print(f"Found {len(cached)} cached search results")

if cached:
    table = build_lookup_table(cached)
    for row in table:
        print(f"\n  {row['cache_key']}: {row['metric']}={row['best_score']:.4f} "
              f"(budget={row['budget']}, {row['model']}, {row['n_reps']} reps)")
        for bgt in sorted(row['by_budget'].keys()):
            info = row['by_budget'][bgt]
            print(f"    budget={bgt}: {info['score']:.4f} [{info['model']}/{info['allocation']}]")

In [ ]:
# Export lookup table to JSON for publishing
if cached:
    output_path = Path('..') / 'published_best_hybrids.json'
    with open(output_path, 'w') as f:
        json.dump(table, f, indent=2, default=str)
    print(f"Exported to {output_path}")

## 3. Apply a Blueprint to New Data

Given a blueprint (from a search or lookup table), create the hybrid representation for your data. This is useful for:
- Applying a published recipe to new data
- Creating train/test splits with identical feature selection

In [ ]:
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

# Use the best blueprint from the search above
bp = result.best_overall['blueprint']
print(f"Blueprint: budget={bp.budget}, alloc={bp.allocation_method}, "
      f"imp={bp.importance_method}, eval={bp.internal_evaluator}")

# Create train/test split
gkf = GroupKFold(n_splits=5)
train_idx, test_idx = next(gkf.split(labels, labels, groups=groups))

train_reps = {k: v[train_idx] for k, v in reps.items()}
test_reps = {k: v[test_idx] for k, v in reps.items()}
y_train, y_test = labels[train_idx], labels[test_idx]

# Create hybrid on training data
X_train, feature_info = create_hybrid(
    train_reps, y_train,
    allocation_method=bp.allocation_method,
    importance_method=bp.importance_method,
    internal_evaluator=bp.internal_evaluator,
    budget=bp.budget,
)

# Apply same feature selection to test data
X_test = apply_feature_selection(test_reps, feature_info)

print(f"\nHybrid shape: train={X_train.shape}, test={X_test.shape}")
print(f"Features per rep:")
for rep_name, info in feature_info.items():
    if isinstance(info, dict) and 'n_selected' in info:
        print(f"  {rep_name}: {info['n_selected']}")

In [ ]:
# Train a model on the hybrid representation
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train_s, y_train)
y_pred = model.predict(X_test_s)

r2 = r2_score(y_test, y_pred)
print(f"R2 on held-out fold: {r2:.4f}")

## 4. Bring Your Own Representations

KIRBy works with any pre-computed feature matrices. You don't need to use the built-in loaders.

In [ ]:
# Example: combine your own feature sets
# Each representation must be (n_samples, n_features) with the same n_samples

n_samples = 500
my_reps = {
    'fingerprints': np.random.rand(n_samples, 2048),
    'descriptors':  np.random.rand(n_samples, 200),
    'embeddings':   np.random.rand(n_samples, 768),
}
my_labels = np.random.rand(n_samples)
my_groups = np.random.randint(0, 50, n_samples)  # e.g., scaffold clusters

my_result = find_best_hybrid(
    base_reps=my_reps,
    labels=my_labels,
    groups=my_groups,
    budgets=[50, 200, 500],
    speed='fast',
    metric='r2',
)

print(f"Best: {my_result.best_overall['score_mean']:.4f} "
      f"(budget={my_result.best_overall['budget']})")
print(f"\nBest by budget:")
for bgt, r in sorted(my_result.best_by_budget.items()):
    print(f"  {bgt}: {r['score_mean']:.4f}")